# MGMT298D: Science and Strategy of AI## Week 5: Neural Networks### UCLA Anderson School of Management

## Imports

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import load_breast_cancerfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import confusion_matrix, classification_report, accuracy_scorefrom tensorflow.keras import Sequentialfrom tensorflow.keras.layers import Dense, Dropout, BatchNormalizationfrom tensorflow.keras.optimizers import Adamfrom tensorflow.keras.callbacks import EarlyStoppingfrom xgboost import XGBClassifierimport warningswarnings.filterwarnings('ignore')sns.set_style('whitegrid')%matplotlib inline

## Load Data

In [ ]:
# Load Wisconsin Breast Cancer datasetcancer = load_breast_cancer()X = pd.DataFrame(cancer.data, columns=cancer.feature_names)y = pd.Series(cancer.target, name='diagnosis')print(f"Dataset shape: {X.shape}")print(f"\nClass distribution:")print(y.value_counts().to_dict())print(f"\nFirst few rows:")X.head()

## Train/Test Split

In [ ]:
# Split data: 80/20 with stratificationX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)# Standardize featuresscaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)print(f"Training set size: {X_train_scaled.shape[0]}")print(f"Test set size: {X_test_scaled.shape[0]}")print(f"Number of features: {X_train_scaled.shape[1]}")

## Simple Neural Network

In [ ]:
# Single hidden layer (32 units, relu)model_simple = Sequential([    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),    Dense(1, activation='sigmoid')])model_simple.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])history_simple = model_simple.fit(X_train_scaled, y_train, epochs=50, batch_size=32,                                    validation_split=0.15, verbose=0)# Evaluate on test sety_pred_simple = (model_simple.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()acc_simple = accuracy_score(y_test, y_pred_simple)print(f"Simple NN Test Accuracy: {acc_simple:.4f}")

## Training Curves — Simple NN

In [ ]:
# Side-by-side training curvesfig, axes = plt.subplots(1, 2, figsize=(12, 4))axes[0].plot(history_simple.history['accuracy'], label='Train', linewidth=2)axes[0].plot(history_simple.history['val_accuracy'], label='Val', linewidth=2)axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Accuracy')axes[0].set_title('Simple NN: Accuracy')axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].plot(history_simple.history['loss'], label='Train', linewidth=2)axes[1].plot(history_simple.history['val_loss'], label='Val', linewidth=2)axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Loss')axes[1].set_title('Simple NN: Loss')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Deep Neural Network

In [ ]:
# Three hidden layers (64, 32, 16), relu, no regularizationmodel_deep = Sequential([    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),    Dense(32, activation='relu'),    Dense(16, activation='relu'),    Dense(1, activation='sigmoid')])model_deep.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])history_deep = model_deep.fit(X_train_scaled, y_train, epochs=50, batch_size=32,                                validation_split=0.15, verbose=0)# Evaluate on test sety_pred_deep = (model_deep.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()acc_deep = accuracy_score(y_test, y_pred_deep)print(f"Deep NN Test Accuracy: {acc_deep:.4f}")

## Regularized Neural Network

In [ ]:
# Three hidden layers with Dropout(0.3) and BatchNormalizationmodel_reg = Sequential([    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),    BatchNormalization(),    Dropout(0.3),    Dense(32, activation='relu'),    BatchNormalization(),    Dropout(0.3),    Dense(16, activation='relu'),    BatchNormalization(),    Dropout(0.3),    Dense(1, activation='sigmoid')])model_reg.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])# EarlyStopping callbackearly_stop = EarlyStopping(patience=15, restore_best_weights=True)history_reg = model_reg.fit(X_train_scaled, y_train, epochs=75, batch_size=32,                              validation_split=0.15, callbacks=[early_stop], verbose=0)# Evaluate on test sety_pred_reg = (model_reg.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()acc_reg = accuracy_score(y_test, y_pred_reg)print(f"Regularized NN Test Accuracy: {acc_reg:.4f}")

## Training Curves — Regularized NN

In [ ]:
# Side-by-side training curves highlighting regularization smoothingfig, axes = plt.subplots(1, 2, figsize=(12, 4))axes[0].plot(history_reg.history['accuracy'], label='Train', linewidth=2)axes[0].plot(history_reg.history['val_accuracy'], label='Val', linewidth=2)axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Accuracy')axes[0].set_title('Regularized NN: Accuracy')axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].plot(history_reg.history['loss'], label='Train', linewidth=2)axes[1].plot(history_reg.history['val_loss'], label='Val', linewidth=2)axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Loss')axes[1].set_title('Regularized NN: Loss')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Confusion Matrix — Regularized NN

In [ ]:
# Compute and visualize confusion matrixcm = confusion_matrix(y_test, y_pred_reg)plt.figure(figsize=(7, 5))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',             xticklabels=['Benign', 'Malignant'],            yticklabels=['Benign', 'Malignant'],            cbar_kws={'label': 'Count'})plt.xlabel('Predicted')plt.ylabel('Actual')plt.title('Confusion Matrix — Regularized NN')plt.tight_layout()plt.show()print(classification_report(y_test, y_pred_reg, target_names=['Benign', 'Malignant']))

## XGBoost Baseline

In [ ]:
# Train XGBoost classifier for comparisonxgb_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, verbosity=0)xgb_model.fit(X_train_scaled, y_train)y_pred_xgb = xgb_model.predict(X_test_scaled)acc_xgb = accuracy_score(y_test, y_pred_xgb)print(f"XGBoost Test Accuracy: {acc_xgb:.4f}")

## Model Comparison

In [ ]:
# Compare test accuracy across all modelsmodels = ['Simple NN', 'Deep NN', 'Regularized NN', 'XGBoost']accuracies = [acc_simple, acc_deep, acc_reg, acc_xgb]plt.figure(figsize=(8, 5))bars = plt.bar(models, accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'], alpha=0.8)plt.ylabel('Test Accuracy')plt.title('Model Comparison — Breast Cancer Diagnosis')plt.ylim([0.9, 1.0])# Add value labels on barsfor bar, acc in zip(bars, accuracies):    height = bar.get_height()    plt.text(bar.get_x() + bar.get_width()/2., height,             f'{acc:.4f}', ha='center', va='bottom', fontsize=11)plt.tight_layout()plt.show()

## Effect of Hidden Layer Size

In [ ]:
# Test single-hidden-layer networks with different sizeshidden_sizes = [8, 16, 32, 64, 128]accuracies_by_size = []for size in hidden_sizes:    model_size = Sequential([        Dense(size, activation='relu', input_shape=(X_train_scaled.shape[1],)),        Dense(1, activation='sigmoid')    ])        model_size.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])    model_size.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_split=0.15, verbose=0)        y_pred_size = (model_size.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()    acc_size = accuracy_score(y_test, y_pred_size)    accuracies_by_size.append(acc_size)# Plot resultsplt.figure(figsize=(8, 5))plt.plot(hidden_sizes, accuracies_by_size, marker='o', linewidth=2, markersize=8, color='#2ca02c')plt.xlabel('Hidden Layer Size')plt.ylabel('Test Accuracy')plt.title('Single-Hidden-Layer Network: Size vs Accuracy')plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Hidden Size vs Accuracy:")for size, acc in zip(hidden_sizes, accuracies_by_size):    print(f"  Size {size:3d}: {acc:.4f}")